# Conversational RAG- Memory + Retrieval

**Module 8 · Lesson 8.8**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Query Reformulation - The Missing Piece
- Full Conversational RAG - The Whole Loop
- Memory Management - Sliding Window vs Summarization
- Production Chatbot - Everything Together
- LangChain & LangGraph - Three Eras, One Recommendation
- LlamaIndex ChatEngine - One Parameter, Full Memory

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy sentence-transformers langchain langgraph llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai deepeval google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Pronoun That Broke Retrieval

> 💡 **Info**
>
> A support bot answers cleanly: “What courses does Netsetos offer?” → “GenAI, Data Science, Cloud, and Full-Stack.” Then the user types the most natural follow-up in the world: **“How much does *it* cost?”** The retriever embeds the literal string “how much does it cost”, searches the vector store, and returns… nothing useful. There is no chunk about “it”. A human reads “it” as “the GenAI course” because they remember the last turn; the retriever has no memory and no way to resolve the pronoun. **Conversational RAG closes that gap: before retrieval, it rewrites the follow-up into a standalone question using the conversation history - “How much does the Netsetos GenAI course cost?” - and then manages that history so it never overflows the context window.** This lesson builds it by hand, then tours the 2026 stack (LangGraph, LlamaIndex) that makes it shippable.

Single-turn RAG, one follow-up in:

### 🎯 What you will be able to do after this lesson

- **Build** conversational RAG by hand - query reformulation (coreference resolution), the reformulate→retrieve→generate→remember loop, and memory across the six architectures (buffer, window, summary, hybrid, entity, vector) under a token budget.

- **Compare** the 2026 stacks: LangGraph checkpointer memory + `create_agent` vs LlamaIndex `condense_plus_context`, and the three LangChain eras.

- **Operate** production chatbots: session persistence, RPM+TPM rate limiting, layered guardrails, multi-user isolation, and streaming.

- **Evaluate** multi-turn RAG with DeepEval’s `ConversationalTestCase` and the three failure modes single-turn metrics miss.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1–8.4 (embeddings, chunking, vector + hybrid retrieval - recalled below), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install google-genai numpy` for the by-hand steps (plus `langchain langgraph langchain-google-genai`, `llama-index`, and `deepeval` for the stack steps). Every block uses the current **google-genai** SDK: pre-2025 tutorials import the retired `google.generativeai` and call dead models, and error on today’s stack.

## 60-Second Warm-Up: What You Already Know

Three recalls, all load-bearing today. Click a card to check yourself.

> 💬 **Analogy**
>
> **Single-turn RAG is texting a stranger; conversational RAG is texting a friend.** The stranger has no memory of your last message - every text must be self-contained. The friend remembers: say “how much is it?” and they know “it” is the thing you were just discussing. Same retrieval engine underneath; the difference is that the friend keeps a running memory of the conversation and resolves your shorthand against it.
> **Where the analogy breaks down:** your friend’s memory is effortless and unbounded; the chatbot’s is neither. Every remembered turn costs tokens in a finite context window, so the system must actively DECIDE what to keep verbatim, what to summarize, and what to drop - the memory-management problem steps 3 and 7 confront. A friend never forgets on purpose; a production chatbot must.

> 💡 **Info**
>
> ⚠️ Misconception: “just stuff the whole chat history into the prompt” (and “a big context window makes memory management unnecessary”)
> Both wrong. Concatenating the full history works for five turns and breaks at fifty: you blow the context window, pay for every token every turn, and models lose factual precision as the prompt approaches their limit - the *lost-in-the-middle* effect, where a model reliably uses facts at the very start and end of a long prompt but tends to overlook those buried in the middle. A 200K or 2M-token model raises the ceiling but does not remove it - cost scales with tokens sent, and long prompts still degrade. So the production answer is not “send everything” - it is **managed memory**: keep recent turns verbatim, summarize or vector-index the rest, and hold the whole thing under a token budget. Memory management is not a workaround for small models; it is a permanent engineering requirement.

## Build One: Conversational RAG by Hand

Steps 1–4 build conversational RAG from scratch with nothing but the google-genai SDK: reformulate a follow-up, run the full loop, manage memory two ways, then assemble a production chatbot. Building it once without a framework is what makes every later tool (LangGraph, LlamaIndex) legible - they are all industrial versions of these four moves.

The Conversational RAG loop (two steps more than basic RAG)

---

## 🎯 Concept 1: Query Reformulation - The Missing Piece

### Query Reformulation - The Missing Piece

Use the LLM to rewrite a follow-up into a standalone question BEFORE retrieval.

**Why this is step 1:** reformulation is the single change that turns a Q&A box into a chatbot. Retrieval embeds whatever string you hand it; hand it a pronoun and you get garbage. Resolve the pronoun first - against the conversation history - and every downstream step just works. Every framework in this lesson automates exactly this move, so writing it by hand is what makes them legible.

> 🗨️ **Analogy**
>
> **Reformulation is an interpreter who fills in what you left unsaid.** You point and say “how much is *it*?”; the interpreter, who heard the whole conversation, relays the complete sentence to someone who just walked in: “How much is the Netsetos GenAI course?” The listener (the retriever) never hears your shorthand - only the fully-spelled-out question.
> **Where the analogy breaks down:** a human interpreter rarely guesses wrong; an LLM can resolve “it” to the wrong entity when several are in play (“the GenAI course” vs “the refund”), silently sending a confident but wrong standalone query downstream. That is why production systems log the reformulated query - so a bad rewrite is visible, not buried.

A user says “Tell me more about *it*.” You embed that exact string and search the vector store. What comes back?

- **reformulate** takes the follow-up plus the recent history and asks the LLM for a standalone rewrite.

- If there is no history yet (first turn), it returns the message unchanged - nothing to resolve.

- The rewritten string is what you embed and retrieve on; the original is what you show the user.

**📝 `01_reformulation.py`** - *google-genai*

In [ ]:
# QUERY REFORMULATION - resolve pronouns + context into a standalone question
# pip install google-genai
from google import genai

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

class QueryReformulator:
    def reformulate(self, follow_up, history):
        if not history:
            return follow_up          # first turn: nothing to resolve
        hist = "\n".join(f"{m['role']}: {m['content']}" for m in history[-6:])
        prompt = (
            "Rewrite the follow-up as a STANDALONE question using the history.\n"
            "Resolve every pronoun (it, that, they, this). Return ONLY the question.\n\n"
            f"History:\n{hist}\n\nFollow-up: {follow_up}\n\nStandalone question:"
        )
        return client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt).text.strip()

# Demo: same history, three follow-ups that lean on it
reformulator = QueryReformulator()
history = [
    {"role": "user", "content": "What courses does Netsetos offer?"},
    {"role": "assistant", "content": "Netsetos offers GenAI, Data Science, Cloud, and Full-Stack courses."},
]
for f in ["How much does it cost?", "Can I get a refund on that?", "What are the prerequisites?"]:
    print(f"  follow-up:   {f!r}")
    print(f"  standalone:  {reformulator.reformulate(f, history)!r}\n")

# Output:
#   follow-up:   'How much does it cost?'
#   standalone:  'How much does the Netsetos GenAI course cost?'
#   follow-up:   'Can I get a refund on that?'
#   standalone:  'What is the refund policy for the Netsetos GenAI course?'
#   follow-up:   'What are the prerequisites?'   (already standalone, returned ~as-is)

#### 💡 What just happened

⚡ What just happened?“How much does it cost?” became “How much does the Netsetos GenAI course cost?” by resolving “it” from history. That standalone query can now be embedded and searched; the pronoun version could not. The tradeoff: reformulation adds one LLM call of latency per turn - real cost, but the alternative (garbage retrieval on every follow-up) is worse. Production systems skip the call when there is no history, and log the rewrite so a wrong resolution is visible.

> 💡 **Info**
>
> ⛔ Anti-pattern: “just concatenate the history and retrieve on the whole thing”
> A tempting shortcut is to skip reformulation and instead paste the entire conversation in front of the follow-up, then embed that blob. This is the **wrong way**, and it **fails because** the embedding of a long, multi-topic blob is a blurry average of everything discussed - it retrieves chunks vaguely related to the whole chat, not the sharp answer to the actual follow-up. It also gets more expensive and blurrier every turn. Do not do this: resolve the pronoun into ONE clean standalone question and retrieve on that. Reformulation is a scalpel; history-concatenation is a smudge.

- Pick a follow-up and watch the pronoun resolve against the visible history into a standalone query.
- Toggle history off to see the retriever receive the raw pronoun instead.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Full Conversational RAG - The Whole Loop

### Full Conversational RAG - The Whole Loop

Reformulate → retrieve → generate → remember, in one class.

**Why this is step 2:** step 1 fixed retrieval; now wire it into the whole loop so state actually accumulates. The class below is the smallest thing that deserves the name “chatbot” - it embeds documents once, reformulates each turn, retrieves on the standalone query, generates with both context and recent history, and appends the turn to memory. Every later framework is this loop with the parts swapped for production components.

> 🔄 **Analogy**
>
> **It is a receptionist with a notepad.** Each time you speak, they (1) glance at the notepad to understand your shorthand, (2) walk to the filing cabinet for the right folder, (3) answer you, and (4) jot the exchange on the notepad before the next question. The notepad is memory; the cabinet is the vector store; the glancing-first is reformulation.
> **Where the analogy breaks down:** the receptionist’s notepad is unbounded, but this one has a hard page limit (the context window). The demo caps history at `max_history` turns and simply drops the oldest - fine for a demo, lossy for a real 50-turn conversation. Step 3 replaces that crude cap with real memory strategies.

In the loop, should you retrieve using the user’s *raw* follow-up or the *reformulated* standalone query?

- **_reformulate** (step 1) turns the follow-up into a standalone query.

- **_retrieve** embeds that query with `gemini-embedding-001` and ranks chunks by cosine similarity.

- **generate** answers from the retrieved context plus the last few turns of history.

- **remember** appends the turn and truncates to `max_history` exchanges.

**📝 `02_conversational_rag.py`** - *google-genai*

In [ ]:
# CONVERSATIONAL RAG - the complete reformulate->retrieve->generate->remember loop
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()

def embed(text):
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

class ConversationalRAG:
    def __init__(self, docs, max_history=10):
        self.history, self.max_history = [], max_history
        self.chunks = docs
        self.embs = [embed(d["text"]) for d in docs]   # index once

    def _reformulate(self, query):
        if not self.history: return query
        hist = "\n".join(f"{m['role']}: {m['content'][:100]}" for m in self.history[-6:])
        return client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Rewrite as a standalone question, resolving pronouns.\n{hist}\nFollow-up: {query}\nStandalone:").text.strip()

    def _retrieve(self, query, k=3):
        qe = embed(query)
        order = sorted(range(len(self.chunks)), key=lambda i: -float(np.dot(qe, self.embs[i])))
        return [self.chunks[i] for i in order[:k]]   # rank by index: O(n log n), duplicate-safe

    def chat(self, user_msg):
        standalone = self._reformulate(user_msg)                 # 1. reformulate
        docs = self._retrieve(standalone)                        # 2. retrieve on the clean query
        context = "\n".join(d["text"] for d in docs)
        hist = "\n".join(f"{m['role']}: {m['content'][:80]}" for m in self.history[-4:])
        answer = client.models.generate_content(model="gemini-2.5-flash",  # 3. generate
            contents=f"Answer from context only, conversationally.\n\nContext:\n{context}\n\n{hist}\nUser: {user_msg}\nAssistant:").text.strip()
        self.history += [{"role":"user","content":user_msg},        # 4. remember
                         {"role":"assistant","content":answer}]
        self.history = self.history[-self.max_history*2:]
        return {"answer": answer, "standalone": standalone,
                "sources": [d.get("source","?") for d in docs]}

docs = [
    {"text":"Netsetos offers GenAI, Data Science, Cloud, and Full-Stack courses.","source":"about"},
    {"text":"GenAI course costs 14999 rupees: lifetime access, Discord, weekly live sessions.","source":"pricing"},
    {"text":"Refund policy: full refund within 7 days, 50% for 8-30 days, none after 30.","source":"refund"},
    {"text":"Prerequisites: basic Python and high-school math. No ML experience needed.","source":"prereqs"},
]
bot = ConversationalRAG(docs)
for msg in ["What courses does Netsetos offer?", "How much does the GenAI one cost?",
            "What if I don't like it?", "How long do I have for that?"]:
    r = bot.chat(msg)
    print(f"You: {msg}\nBot: {r['answer'][:90]}\n  [standalone: {r['standalone'][:55]!r} | {r['sources']}]\n")

# Output:
# You: How much does the GenAI one cost?
# Bot: The Netsetos GenAI course costs 14,999 rupees ...
#   [standalone: 'How much does the Netsetos GenAI course cost?' | ['pricing']]
# You: What if I don't like it?
#   [standalone: 'What is the refund policy for the GenAI course?' | ['refund']]

#### 💡 What just happened

⚡ What just happened? Four progressively vaguer follow-ups, each resolved and routed to the right document: “the GenAI one” → pricing, “what if I don’t like it?” → refund. Without reformulation, turns 2–4 would all fail retrieval on their pronouns. The tradeoff baked in here: the crude `max_history` cap keeps the prompt bounded but silently forgets old turns - cheap and lossy. Step 3 makes that memory decision a real strategy instead of a truncation.

- Scrub through a single turn and watch it pass reformulate → retrieve → generate → remember.
- See which document each standalone query lights up in the store.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Memory Management - Sliding Window vs Summarization

### Memory Management - Sliding Window vs Summarization

Conversations reach 50+ turns. You cannot send them all. Two strategies.

**Why this is step 3:** the crude cap from step 2 throws away context the moment the conversation gets interesting. Real systems choose HOW to forget. The two foundational strategies - keep the last N turns verbatim (window), or compress the old ones into a running summary (summarize) - are the atoms every framework’s memory is built from. Step 7 expands this to six architectures; here you build the two that matter most.

> 📝 **Analogy**
>
> **Window memory is a whiteboard; summary memory is meeting minutes.** A whiteboard holds only what fits - write a new line and the oldest scrolls off; you always see the most recent thinking, verbatim, but early context is gone. Minutes are different: a scribe periodically condenses the discussion into a few bullet points, so nothing is fully lost, but the detail is.
> **Where the analogy breaks down:** a scribe writing minutes is free; the summarizer here is an extra LLM call with its own latency and cost, and it can drop the exact detail a later turn needs (“the price you quoted in turn 3”). Summarization trades tokens for an information bottleneck - which is why production often keeps recent turns verbatim AND summarizes the tail (the hybrid you meet in step 7).

A 40-turn support chat. Window memory (last 6 turns). The user refers to a detail from turn 2. What happens?

- **window**: `get_context` returns just the last `window_size` messages - O(1), lossy on old turns.

- **summarize**: when history exceeds the window, the overflow is condensed by an LLM into `self.summary`, prepended as a system note; recent turns stay verbatim.

- Both keep the prompt bounded; they differ in what they sacrifice - old detail (window) vs exactness (summary).

**📝 `03_memory.py`** - *google-genai*

In [ ]:
# MEMORY MANAGEMENT - sliding window (fast, lossy) vs summarization (smart, costs a call)
# pip install google-genai
from google import genai

client = genai.Client()

class ChatMemory:
    def __init__(self, strategy="window", window_size=6):
        self.strategy, self.window_size = strategy, window_size
        self.messages, self.summary = [], ""

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})

    def get_context(self):
        if self.strategy == "window":
            return self.messages[-self.window_size:]          # keep the last N, drop the rest
        # summarize: condense everything older than the window into a running summary
        if len(self.messages) > self.window_size:
            old = self.messages[:-self.window_size]
            text = "\n".join(f"{m['role']}: {m['content'][:80]}" for m in old)
            self.summary = client.models.generate_content(model="gemini-2.5-flash",
                contents=f"Summarize this conversation in 2-3 sentences:\n{text}").text.strip()
            self.messages = self.messages[-self.window_size:]
        ctx = list(self.messages)
        if self.summary:
            ctx.insert(0, {"role":"system", "content":f"Earlier: {self.summary}"})
        return ctx

for strat in ["window", "summarize"]:
    mem = ChatMemory(strategy=strat, window_size=4)
    for i in range(8):
        mem.add("user", f"Question {i+1}"); mem.add("assistant", f"Answer {i+1}")
    ctx = mem.get_context()
    print(f"{strat:10s}: {len(ctx)} messages in context"
          + (f" | summary: {mem.summary[:50]}..." if mem.summary else ""))

# Output:
# window    : 4 messages in context
# summarize : 5 messages in context | summary: The user asked several questions about ...

#### 💡 What just happened

⚡ What just happened? After sixteen messages, *window* keeps the last four verbatim and forgets the rest; *summarize* keeps the last four PLUS a one-line summary of the older ones, so the gist survives. The tradeoff is explicit: window is free but forgets early detail; summarize costs an extra LLM call each time it compresses and can lose the precise fact a later turn needs. Neither is universally right - you choose by conversation length and how often users reference far-back turns.

- Drag the window-size slider and watch how many tokens each strategy keeps as turns pile up.
- Flip to summarize and see old turns collapse into a summary line instead of vanishing.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Production Chatbot - Everything Together

### Production Chatbot - Everything Together

System prompt + reformulation + retrieval + memory + source tracking + stats.

**Why this is step 4:** steps 1–3 are the moving parts; this assembles them into one object with the scaffolding a real deployment needs - a system prompt to shape tone and refusals, source tracking so answers are attributable, and per-turn latency stats so you can see reformulation’s cost. It is still under 40 lines, and it is the mental model you carry into the framework steps.

> 🏪 **Analogy**
>
> **Steps 1–3 were parts on a workbench; step 4 bolts them into one machine with a nameplate.** The system prompt is the operating manual (“answer only from context, be a Netsetos assistant”); source tracking is the audit sticker on each output; the stats dict is the dashboard gauge. Same parts - now shippable.
> **Where the analogy breaks down:** a bolted-together machine is finished; this one is deliberately minimal - no persistence across restarts, no rate limits, no guardrails. Those are steps 8’s concerns. Treat step 4 as the reference the frameworks improve on, not a production deployment.

On the FIRST turn of a fresh conversation, does the production chatbot pay for a reformulation LLM call?

- A **system prompt** is prepended to every generation to fix tone and grounding.

- **sources** and a **stats** dict (turns, reformulations, latency) are returned so the caller can log and monitor.

- Everything else is the step-2 loop - reformulate, retrieve, generate, remember.

**📝 `04_production_chatbot.py`** - *google-genai*

In [ ]:
# PRODUCTION CONVERSATIONAL RAG - system prompt + sources + latency stats
# pip install google-genai numpy
import numpy as np, time
from google import genai

client = genai.Client()
def embed(t): return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

class ProductionChatbot:
    def __init__(self, docs, system="You are a Netsetos support assistant. Answer from context only."):
        self.system, self.history = system, []
        self.chunks, self.embs = docs, [embed(d["text"]) for d in docs]
        self.stats = {"turns":0, "reformulated":0}

    def chat(self, msg):
        t0 = time.time()
        q = msg
        if self.history:                                          # reformulate only when there is history
            h = "\n".join(f"{m['role']}: {m['text'][:60]}" for m in self.history[-6:])
            q = client.models.generate_content(model="gemini-2.5-flash",
                contents=f"Rewrite as standalone, resolve pronouns.\n{h}\nFollow-up: {msg}\nStandalone:").text.strip()
            self.stats["reformulated"] += 1
        qe = embed(q)                                             # retrieve on the standalone query
        top = sorted(range(len(self.chunks)), key=lambda i: -float(np.dot(qe, self.embs[i])))[:3]
        ctx = "\n".join(self.chunks[i]["text"] for i in top)
        hist = "\n".join(f"{m['role']}: {m['text'][:60]}" for m in self.history[-4:])
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"{self.system}\n\nContext:\n{ctx}\n\n{hist}\nUser: {msg}\nAssistant:").text.strip()
        self.history += [{"role":"user","text":msg}, {"role":"assistant","text":ans}]
        self.stats["turns"] += 1
        return {"answer":ans, "query":q,
                "sources":[self.chunks[i].get("source","?") for i in top],
                "latency_ms":(time.time()-t0)*1000}

docs = [{"text":"Netsetos offers GenAI, Data Science, Cloud, Full-Stack courses.","source":"about"},
        {"text":"GenAI course: 14999 rupees. Lifetime access, EMI via Razorpay.","source":"pricing"},
        {"text":"Refund: full 7 days, 50% 8-30 days, none after 30. Email support@netsetos.com.","source":"refund"}]
bot = ProductionChatbot(docs)
for msg in ["What GenAI courses do you have?", "How much is it?", "Can I pay monthly?", "What if I want a refund?"]:
    r = bot.chat(msg)
    print(f"You: {msg}\nBot: {r['answer'][:80]}  [{r['latency_ms']:.0f}ms | {r['sources']}]\n")
print(f"Stats: {bot.stats}")

# Output:
# You: How much is it?
# Bot: The GenAI course is 14,999 rupees ...  [512ms | ['pricing']]
# Stats: {'turns': 4, 'reformulated': 3}

#### 💡 What just happened

⚡ What just happened? The same four moves, now with a system prompt, source attribution, and latency stats - enough to log, debug, and monitor. Notice `reformulated: 3`: the first turn skipped reformulation (no history), the other three paid one extra LLM call each. That is the honest cost of conversational RAG. Everything from here replaces a hand-written part with a battle-tested one - but the shape never changes.

## The 2026 Conversational-RAG Stack

You now own the concepts. Steps 5–10 map them onto the tools you will actually ship with: LangGraph and LangChain, LlamaIndex chat engines, the six memory architectures, production hardening, multi-turn evaluation, and India-specific patterns. Same loop - industrial parts.

---

## 🎯 Concept 5: LangChain & LangGraph - Three Eras, One Recommendation

### LangChain & LangGraph - Three Eras, One Recommendation

Deprecated chain → RunnableWithMessageHistory → LangGraph checkpointer memory.

**Why this is step 5:** LangChain is the most common way teams ship conversational RAG, and it has changed twice. Tutorials still teach the deprecated `ConversationalRetrievalChain`; the current recommendation (LangChain 1.0, Oct 2025) is an agent with a retrieval tool and **LangGraph checkpointer memory**. Knowing which era a snippet is from saves you from shipping dead code.

> 📦 **Info**
>
> 🔗 The three eras (know which one a snippet is from)
> - **Era 1 - `ConversationalRetrievalChain` (deprecated, v0.1.17).** Hid its prompts, no streaming/async. If a tutorial imports it, the tutorial is stale.
> - **Era 2 - `create_history_aware_retriever` + `create_retrieval_chain`.** Still valid: reformulates with history, then retrieves. The transitional pattern many codebases still run.
> - **Era 3 - LangGraph (recommended).** Expose retrieval as a `@tool` and let `create_agent` decide when to call it; a **checkpointer** saves the entire graph state per node, keyed by `thread_id`. This is what `RunnableWithMessageHistory` (message-only, legacy) grew into - and it adds time-travel, human-in-the-loop, and fault recovery (see the LangGraph source at [github.com/langchain-ai/langgraph](https://github.com/langchain-ai/langgraph)).

> 💾 **Analogy**
>
> **A LangGraph checkpointer is a video game’s save system; RunnableWithMessageHistory is only an autosave of the chat log.** The chat log remembers what was said. A full save-state remembers the whole world - messages, tool outputs, intermediate results - so you can reload the exact moment, branch from it, or recover after a crash. The `thread_id` is the save slot: one per conversation.
> **Where the analogy breaks down:** a game save is one file on your disk; a production checkpointer is a database (Postgres/Redis) with retention, migration, and privacy obligations - a user’s entire conversation state now lives in your store and falls under data-deletion rules (step 10). Saving state is easy; governing it is the real work.

Two users chat with the same LangGraph agent at the same time. What keeps their histories from mixing?

- **@tool** wraps your retriever so the agent can call it - pass it a standalone question.

- **create_agent** builds the loop; **InMemorySaver** is the checkpointer (swap `PostgresSaver` in production).

- Each `invoke` passes a `thread_id` in the config; the checkpointer loads that conversation’s state, so “it” resolves from saved history automatically.

**📝 `05_langgraph_rag.py`** - *LangGraph*

In [ ]:
# CONVERSATIONAL RAG, THE 2026 WAY - agent + retrieval tool + checkpointer memory
# pip install langchain langgraph langchain-google-genai
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
# production: from langgraph.checkpoint.postgres import PostgresSaver

@tool
def search_courses(query: str) -> str:
    """Search Netsetos course docs. Pass a STANDALONE question (no pronouns)."""
    docs = retriever.invoke(query)                 # your vector store from Lesson 8.1
    return "\n\n".join(d.page_content for d in docs)

checkpointer = InMemorySaver()                      # thread_id -> saved graph state
agent = create_agent(
    model=init_chat_model("google_genai:gemini-2.5-flash"),
    tools=[search_courses],
    system_prompt="Answer only from search_courses results. Cite the source.",
    checkpointer=checkpointer,
)

cfg = {"configurable": {"thread_id": "user-42"}}    # one thread per conversation
for msg in ["What GenAI courses do you have?", "How much is it?", "Can I pay monthly?"]:
    out = agent.invoke({"messages": [{"role":"user", "content": msg}]}, cfg)
    print(f"You: {msg}\nBot: {out['messages'][-1].content[:80]}\n")

# Output:
# You: What GenAI courses do you have?
# Bot: Netsetos offers a GenAI Engineering course ... (source: about)
# You: How much is it?     <- 'it' resolved from thread state; agent re-queries the tool
# Bot: The GenAI course is 14,999 rupees ... (source: pricing)
# You: Can I pay monthly?   <- 'pay' still about the GenAI course
# Bot: Yes - EMI is available via Razorpay ... (source: pricing)

#### 💡 What just happened

⚡ What just happened? The agent handled “How much is it?” without any hand-written reformulation: the checkpointer restored the conversation’s saved state for `thread_id="user-42"`, so the model resolved “it” and re-called `search_courses` with a standalone query on its own. The tradeoff vs the by-hand version: you give up fine control over the exact reformulation prompt in exchange for automatic memory, multi-user isolation by `thread_id`, and the recovery/time-travel a checkpointer enables. For most teams that is the right trade - but you now know exactly what it is doing underneath.

- Switch between two `thread_id`s and watch each keep its own isolated saved state.
- Step through nodes to see a checkpoint written after every node boundary.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: LlamaIndex ChatEngine - One Parameter, Full Memory

### LlamaIndex ChatEngine - One Parameter, Full Memory

The 8.6 forward hook: `as_chat_engine` gives condensing + retrieval + memory.

**Why this is step 6:** LlamaIndex collapses everything you built by hand into a single method call. Where LangGraph gives you an agent to configure, LlamaIndex gives you a chat engine with five ready modes - and the production choice, `condense_plus_context`, does reformulation AND retrieval AND memory in one line. It is the fastest path from “I have documents” to “I have a chatbot.”

> 🍽️ **Analogy**
>
> **The chat engine is a maître d’ with a good memory.** You do not re-introduce yourself between courses; the maître d’ remembers you asked about the tasting menu, so “and how much is *it*?” is understood without repetition. `condense_plus_context` is exactly that: it condenses your follow-up plus history into one standalone question, retrieves for it, and answers - memory included.
> **Where the analogy breaks down:** a maître d’ improvises; the chat mode is fixed. Pick `condense_question` and the engine literally cannot answer “what did I just ask you?” because it only ever sees a rewritten query, not the raw turn. The mode is a real design decision - the comparison table below is the menu.

You pick `chat_mode="condense_question"`. A user asks “what did I just ask you?” Can the engine answer?

| chat_mode | How it works | Best for |
|---|---|---|
| condense_question | Reformulate → query engine. Cannot answer “what did I ask?” | Pure knowledge-base Q&A |
| condense_plus_context | Reformulate + retrieve context + answer; supportssystem_prompt | Production chatbots (recommended) |
| simple | LLM chat with history, no retrieval | General conversation fallback |
| react | Agent decides whether to retrieve | Complex agentic interactions |
| context | Always retrieve, no condense step | Always-retrieve scenarios |

- **Settings** sets the current stack once: `GoogleGenAI` + `GoogleGenAIEmbedding`.

- **as_chat_engine(chat_mode="condense_plus_context")** is the whole feature - condense + retrieve + memory.

- **ChatMemoryBuffer.from_defaults(token_limit=...)** caps history by tokens, counted backward from the most recent turn.

**📝 `06_llamaindex_chat.py`** - *LlamaIndex*

In [ ]:
# LLAMAINDEX CHATENGINE - condense_plus_context is the production mode
# pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")
Settings.embed_model = GoogleGenAIEmbedding(model_name="gemini-embedding-001")

index = VectorStoreIndex.from_documents(documents)          # your docs
chat = index.as_chat_engine(
    chat_mode="condense_plus_context",                        # reformulate AND retrieve
    memory=ChatMemoryBuffer.from_defaults(token_limit=3000),
    system_prompt="You are a Netsetos support assistant. Answer from context only.",
)

print(chat.chat("What GenAI courses do you have?"))
print(chat.chat("How much is it?"))        # 'it' condensed from memory, then retrieved
chat.reset()                                                # clear memory for a new session

# Output:
# Netsetos offers a GenAI Engineering course covering ...
# The GenAI course is priced at 14,999 rupees ...

#### 💡 What just happened

⚡ What just happened? Your entire step-4 chatbot collapsed into `as_chat_engine(chat_mode="condense_plus_context")` plus a memory buffer. The tradeoff vs LangGraph: LlamaIndex is faster to stand up and opinionated (five modes, pick one), while LangGraph gives finer control and durable checkpointing. Both beat hand-rolling - but only because you built it by hand first do you know that `condense_plus_context` is just your reformulate-then-retrieve loop with a token-limited buffer.

---

## 🎯 Concept 7: Six Memory Architectures & the Token Budget

### Six Memory Architectures & the Token Budget

Buffer, window, summary, hybrid, entity, vector - and staying under the ceiling.

**Why this is step 7:** step 3 gave you two memory strategies; production uses six, and picks by conversation shape. More importantly, whichever you pick, you must fit it into a finite context alongside the retrieved documents - that is the token-budget discipline that separates a demo from a system that does not fall over at turn 30.

| Memory type | Mechanism | Best for |
|---|---|---|
| Buffer | Full history verbatim | Short conversations (<15 turns) |
| Window (k=5-10) | Last K exchanges | Task-oriented, bounded interactions |
| Summary | Running LLM-summarized digest | Very long conversations |
| Hybrid summary-buffer | Recent verbatim + older summarized | Best general-purpose (recommended) |
| Entity | Tracks named entities across turns | CRM, medical assistants |
| Vector | Embeds turns, retrieves by similarity | Referencing context from 50+ turns ago |

> 🏦 **Analogy**
>
> **The token budget is a carry-on bag.** Fixed capacity, and four things want in: the system prompt (your travel documents - always packed), recent history (this week’s clothes), retrieved documents (the gear for this trip), and room for the response (space to bring something back). Overpack any one and the bag will not close; you must decide what gets folded small (summarized) and what gets left home (dropped).
> **Where the analogy breaks down:** a bag either closes or it does not - binary. A context window degrades gradually: fill it near the top and the model does not error, it just gets subtly less accurate (lost-in-the-middle). So the rule is not “fit under the limit” but “stay comfortably below it,” leaving headroom rather than packing to the brim.

A user references a table you showed 50 turns ago. Window (k=10) dropped it. Which memory recovers it?

- **ContextBudget** allocates a share of the window to system / history / retrieval / response.

- **fit** checks the running total; on overflow it drops the oldest history first (a real system would summarize it), then trims retrieved chunks - not the other way around.

- Priority order encodes a value judgement: never drop the system prompt, protect response headroom, sacrifice retrieval breadth last.

**📝 `07_token_budget.py Pure`** - *Python*

In [ ]:
# TOKEN BUDGET - allocate the context window, truncate by priority on overflow
# A rough 4-chars-per-token estimate keeps this dependency-free and deterministic.
def est_tokens(text):
    return len(text) // 4

class ContextBudget:
    def __init__(self, window=8000):
        # reserve a share of the window for each part (an example split for an 8K model)
        self.window = window
        self.response_reserve = int(window * 0.25)   # always leave room to answer

    def fit(self, system, history, retrieved):
        budget = self.window - self.response_reserve
        used = est_tokens(system)                       # system prompt: never dropped
        # history and retrieval compete for what is left
        hist = "\n".join(history)
        while est_tokens(hist) > (budget - used) * 0.4 and len(history) > 2:
            history = history[2:]                          # drop oldest turns first (a real system summarizes them - see step 3)
            hist = "[...earlier turns dropped...]\n" + "\n".join(history)
        used += est_tokens(hist)
        docs = []
        for chunk in retrieved:                          # then trim retrieval to fit
            if used + est_tokens(chunk) > budget: break
            docs.append(chunk); used += est_tokens(chunk)
        return {"system":system, "history":hist, "context":docs,
                "tokens_used":used, "headroom":self.window - used}

b = ContextBudget(window=8000)
plan = b.fit(system="You are a Netsetos support assistant.",
             history=[f"user turn {i}: asked about pricing, refunds, and prerequisites in detail "*9 for i in range(20)],
             retrieved=["retrieved policy document chunk with the relevant answer context text "*45 for _ in range(6)])
print(f"tokens used: {plan['tokens_used']} | headroom: {plan['headroom']} | chunks kept: {len(plan['context'])}")
print("history folded:", plan["history"].startswith("[...earlier"))

# Output:
# tokens used: 5426 | headroom: 2574 | chunks kept: 4
# history folded: True   (20 turns dropped down to 14 to fit; 2 of 6 chunks trimmed)

#### 💡 What just happened

⚡ What just happened? The budget kept the system prompt untouched, folded the 20-turn history down to 14 to fit its share, then admitted only 4 of 6 retrieved chunks before it would have eaten the response reserve. That priority order - protect the system prompt and response headroom, drop old history first, sacrifice retrieval breadth last - is the design decision. The tradeoff is visible in the numbers: dropping two chunks trades a little retrieval recall for guaranteed room to answer. (For determinism this demo *drops* old turns; a production system would *summarize* them, exactly as step 3 did.) Hybrid summary-buffer plus a budget like this is the general-purpose production default.

- Drag the sliders to grow history or retrieval and watch the bar approach the ceiling.
- Push past the limit to trigger the priority truncation - history folds before retrieval trims.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: Production - Sessions, Rate Limits, Guardrails, Isolation

### Production - Sessions, Rate Limits, Guardrails, Isolation

What the demos skipped: persistence, throttling, safety, multi-tenancy, streaming.

**Why this is step 8:** everything so far assumed one user, one process, infinite budget, and good-faith input. Production assumes none of that. This step is the checklist between “works on my machine” and “serves thousands of users” - session persistence across restarts, rate limits on both requests and tokens, layered guardrails, and hard isolation between tenants.

> 🏭 **Analogy**
>
> **Steps 1–7 built one shop; step 8 turns it into a franchise.** One shop trusts every customer and never closes; a franchise needs a cash register that survives a power cut (session store), a queue policy so nobody hogs the counter (rate limits), a security guard at the door (guardrails), and separate lockers so customers never see each other’s things (isolation).
> **Where the analogy breaks down:** a security guard makes one judgement; guardrails stack, and stacking has a hidden cost. Five independent guards at roughly 90% accuracy each reject about 41% of legitimate messages between them (compounding false positives). So more guards is not safer past a point - you need each guard at extremely high precision, or the system blocks its own users.

You rate-limit by requests-per-minute only. One user opens a single very long, document-heavy conversation. What breaks?

> ✅ **Info**
>
> 🔒 The production checklist
> - **Sessions:** Redis (hot, sub-ms) + PostgreSQL/DynamoDB (cold, durable). Reload conversation state on restart so a deploy does not wipe every chat.
> - **Rate limits:** enforce BOTH RPM (requests/min) and TPM (tokens/min) - a single long conversation can blow your token budget without tripping a request limit. Token bucket in Redis for smooth traffic.
> - **Guardrails (3 tiers):** rule-based (fast regex PII checks), ML classifiers (toxicity/NER), and LLM-based guards (Llama Guard) - each deeper tier adds latency. Mind the compounding false-positive math above.
> - **Multi-user isolation:** tenant-scoped retrieval with server-side metadata filters; per-user session keys. A context-window bleed between tenants is a security incident, not a bug.
> - **Streaming:** persist the turn to history AFTER the stream finishes, never mid-stream - a dropped connection must not save half an answer.

- **SessionStore** keys history by `session_id` (a dict here; Redis in production) so users never collide.

- **allow** is a token-bucket rate limiter: refill over time, spend on each call, reject when empty.

- The pattern is transport-agnostic - wire it behind FastAPI, and stream with the persist-after-complete rule.

**📝 `08_production.py Pure`** - *Python*

In [ ]:
# PRODUCTION SCAFFOLDING - per-session state + a token-bucket rate limiter
# In production: back SessionStore with Redis, the bucket with a Redis Lua script.
import time
from collections import defaultdict

class SessionStore:
    def __init__(self):
        self._data = defaultdict(list)              # session_id -> history; Redis in prod
    def append(self, session_id, role, text):
        self._data[session_id].append({"role":role, "text":text})
    def history(self, session_id):
        return self._data[session_id]              # isolated per user - no cross-tenant bleed

class RateLimiter:
    """Token bucket: `rate` tokens/sec, burst up to `capacity`."""
    def __init__(self, rate=5, capacity=10):
        self.rate, self.capacity = rate, capacity
        self.tokens, self.ts = capacity, time.time()
    def allow(self, cost=1):
        now = time.time()
        self.tokens = min(self.capacity, self.tokens + (now - self.ts) * self.rate)
        self.ts = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False                              # over budget - return 429 to the caller

store, limiter = SessionStore(), RateLimiter(rate=5, capacity=10)
for i in range(12):
    ok = limiter.allow()
    if ok:
        store.append("user-42", "user", f"message {i}")
    print(f"req {i:2d}: {'served' if ok else 'RATE-LIMITED (429)'}")
print(f"user-42 history length: {len(store.history('user-42'))}")

# Output:
# req  0: served ... req 10: RATE-LIMITED (429)  req 11: RATE-LIMITED (429)
# user-42 history length: 10

#### 💡 What just happened

⚡ What just happened? The bucket served its opening burst up to capacity, then returned 429s until it refilled - and every served turn landed in that user’s isolated history. Swap the dict for Redis and the bucket for a Redis Lua script and this is a real deployment’s spine. The tradeoff at the heart of step 8: every safety layer (a guard, a limit, an isolation check) costs latency and can reject good traffic, so production is a balance - enough protection to be safe, not so much that the system blocks its own users.

---

## 🎯 Concept 9: Multi-Turn Evaluation - What Single-Turn Metrics Miss

### Multi-Turn Evaluation - What Single-Turn Metrics Miss

DeepEval `ConversationalTestCase` and the three multi-turn failure modes.

**Why this is step 9:** you cannot ship what you cannot measure, and single-turn RAG metrics (faithfulness, answer relevancy on one Q&A) miss the failures that only appear across turns. A conversation can be individually-correct on every turn yet drift, repeat, or cross-contaminate over ten. Multi-turn evaluation is how you catch that before users do.

> 🎥 **Analogy**
>
> **Single-turn metrics grade photographs; multi-turn metrics grade the movie.** Each frame can be in focus while the film makes no sense - the plot wanders (drift), a scene repeats (redundant retrieval), or a character suddenly knows something from a deleted scene (cross-turn hallucination). You have to watch the whole reel, in order, to see those.
> **Where the analogy breaks down:** a film critic watches once; an eval harness must run on every code change, cheaply and deterministically, as a CI gate. That is why you build a fixed golden set of multi-turn cases rather than judging ad hoc - the harness has to give the same verdict twice.

Every individual turn scores high on faithfulness, but users complain the bot “loses the thread.” What are you failing to measure?

- **ConversationalTestCase(turns=[...])** holds the whole conversation; each retrieval-bearing `Turn` carries its own `retrieval_context`.

- **TurnRelevancyMetric** slides a window of prior turns to judge whether each reply stays on-thread.

- **TurnFaithfulnessMetric** checks each answer against its turn’s retrieved context - per-turn grounding across the conversation.

**📝 `09_eval.py`** - *DeepEval*

In [ ]:
# MULTI-TURN RAG EVALUATION - conversation-level metrics single-turn RAG misses
# pip install deepeval
from deepeval.test_case import ConversationalTestCase, Turn
from deepeval.metrics import TurnRelevancyMetric, TurnFaithfulnessMetric
from deepeval import evaluate

test = ConversationalTestCase(turns=[
    Turn(role="user", content="What GenAI courses do you have?"),
    Turn(role="assistant", content="The Netsetos GenAI Engineering course.",
         retrieval_context=["Netsetos offers a GenAI Engineering course."]),
    Turn(role="user", content="How much is it?"),
    Turn(role="assistant", content="It costs 14,999 rupees.",
         retrieval_context=["GenAI course: 14999 rupees, lifetime access."]),   # 'it' resolved
])

evaluate(test_cases=[test],
         metrics=[TurnRelevancyMetric(), TurnFaithfulnessMetric()])

# Output:
# Metrics Summary
#   - Turn Relevancy: 1.00 (passed)   reply stays on-thread across turns
#   - Turn Faithfulness: 1.00 (passed) each answer grounded in its turn's context

#### 💡 What just happened

⚡ What just happened? DeepEval scored the CONVERSATION, not two isolated answers - `TurnRelevancy` slid a window over prior turns to confirm the reply stayed on-thread, and `TurnFaithfulness` grounded each answer in its own turn’s retrieval. Those catch the three multi-turn failure modes single-turn metrics cannot: **context drift** (retrieval wanders off-topic), **redundant retrieval** (same chunks every turn), and **cross-turn hallucination** (mixing facts from different turns). Wire this into CI with a golden set and every change is gated - the discipline Lesson 8.11 makes the whole subject.

- Advance turns and watch retrieval precision drift as the topic wanders.
- Toggle memory on to see reformulation pull precision back up.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 10: India Conversational RAG - Hinglish, Multilingual, Compliance

### India Conversational RAG - Hinglish, Multilingual, Compliance

Code-switching users, Indic embeddings, and the DPDP + RBI obligations.

**Why this is step 10:** a chatbot for Indian users faces problems the English-only demos never surface - 250M+ people code-switch between Hindi and English mid-sentence (“refund *kaise* milega?”), Devanagari tokenizes far heavier than English, and a banking or fintech deployment sits under real regulation. Getting these right is the difference between a demo and a product that ships in India.

> 🇮🇳 **Analogy**
>
> **Serving Hinglish is hiring a bilingual receptionist, not a translator.** A translator converts one whole language to another; your users mix both in a single sentence and switch without warning. A bilingual receptionist handles the mix natively - which is what multilingual embeddings (BGE-m3, MuRIL) and a detect-then-transliterate pipeline give you, versus bolting an English-only model onto translated text.
> **Where the analogy breaks down:** a receptionist just needs to understand you; a regulated chatbot must also PROVE what it did - log every turn, record consent, allow erasure. Comprehension is the easy half; the audit trail and consent management the RBI and DPDP rules require are the half that has to be designed in from turn one, not added later.

A user types “Refund *kaise* milega?” (Romanized Hinglish). Your English-only embedder handles it…

> 📦 **Info**
>
> 🌐 The India stack
> - **Hinglish:** 250M+ code-switching speakers, no standard Romanized-Hindi spelling; Devanagari tokenizes roughly 3× heavier than English, so budgets and cost shift. Pipeline: language ID (fastText) → Unicode NFC normalize → transliterate Roman→Devanagari (IndicXlit) → embed.
> - **Embeddings:** BGE-m3 (strong open multilingual), MuRIL and IndicBERT (Indic-tuned), IndicTrans2 for translation (22 languages), Bhashini APIs for language services.
> - **Multilingual sessions:** store original text + language tag + an English translation; reply in the user’s detected language.
> - **In production:** Indian banking assistants like **HDFC Bank EVA**, **SBI SIA**, and **ICICI iPal** show that multilingual conversational banking bots ship and scale in India (several began as intent/NLU systems and are moving toward retrieval-grounded answers), under the regulations below.
> - **Compliance (a recommendations-and-rules landscape - verify exact instruments and live dates before shipping):** RBI **FREE-AI** is a *recommendations framework* (released 13 Aug 2025; 7 “Sutras”, 6 pillars, 26 recommendations - spec at [rbidocs.rbi.org.in](https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FREEAIR130820250A24FF2D4578453F824C72ED9F5D5851.PDF)), not binding regulation; it sets the expected governance - board-approved AI policy, audit trails, explainability, human-in-the-loop, incident reporting - that becomes obligation as the RBI operationalizes it. RBI’s **digital-banking-channel rules** increasingly require explicit, recorded customer consent to enable a new digital channel (confirm the exact instrument and its chatbot scope). The **DPDP Act 2023 + DPDP Rules 2025** (Rules notified 13–14 Nov 2025) phase in most substantive duties - notice/consent, erasure-on-request, breach notification - over an ~18-month transition (commencement around 2027; the Data Protection Board machinery is live earlier). Treat consent, audit trails, and erasure as design constraints from turn one.

- Detect the language of each turn (fastText LID in production; a stub here).

- Transliterate Romanized Hindi to Devanagari so the embedder sees canonical script.

- Embed with a multilingual model (BGE-m3) so Hindi and English land in the same space.

**📝 `10_india.py`** - *Multilingual*

In [ ]:
# INDIA CONVERSATIONAL RAG - a code-switch-aware preprocessing pipeline
# pip install sentence-transformers  (BGE-m3 handles Hindi + English in one space)
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3")     # multilingual, 100+ languages

def detect_lang(text):
    # production: fastText LID; stub keys off a Devanagari codepoint range
    return "hi" if any("ऀ" <= c <= "ॿ" for c in text) else "en-or-roman"

def preprocess(turn):
    lang = detect_lang(turn)
    # production: if Romanized Hindi, transliterate to Devanagari with IndicXlit here
    return {"text": turn, "lang": lang, "embedding": embedder.encode(turn)}

for turn in ["What is the refund policy?", "Refund kaise milega?", "रिफंड कैसे मिलेगा?"]:
    r = preprocess(turn)
    print(f"{r['lang']:12s} | dim={len(r['embedding'])} | {turn}")

# Output:
# en-or-roman  | dim=1024 | What is the refund policy?
# en-or-roman  | dim=1024 | Refund kaise milega?
# hi           | dim=1024 | रिफंड कैसे मिलेगा?

#### 💡 What just happened

⚡ What just happened? All three phrasings of the same question - English, Romanized Hinglish, and Devanagari - embed into the same 1024-dim BGE-m3 space, so they can all retrieve the one refund document. The tradeoff India forces: multilingual coverage plus a detect-transliterate step adds pipeline stages and heavier tokenization (cost), but it is the only way to serve code-switching users without shipping an English-only bot that fails half your market. And the compliance layer - audit trails, recorded consent, erasure - is not optional garnish; under RBI FREE-AI and the DPDP rules it is a design constraint from turn one.

## Putting It Together

You turned single-turn RAG into a chatbot by adding two steps to the pipeline - *reformulate* before retrieval and *remember* after generation - then hardened it: managed memory under a token budget, the LangGraph and LlamaIndex stacks, production concerns, multi-turn evaluation, and India’s language and compliance realities. The through-line: *a conversation is state, and every hard problem here is a decision about what state to keep, how to fit it in a finite window, and how to prove what you did with it.*

> 📦 **Info**
>
> 🔑 Key takeaways
> - **Reformulation is the missing piece.** Retrieval embeds the query string, so a pronoun retrieves garbage; resolve it against history into a standalone question first.
> - **The loop is basic RAG + 2.** Reformulate → retrieve → generate → remember. Everything else swaps a hand-written part for an industrial one.
> - **Memory is a design decision, not a default.** Window vs summary vs vector vs hybrid, chosen by conversation length and how far back users reference - all under a token budget with headroom.
> - **Know the LangChain era.** Deprecated `ConversationalRetrievalChain` → transitional history-aware retriever → recommended LangGraph checkpointer + `create_agent`, isolated by `thread_id`.
> - **Evaluate the movie, not the frames.** Multi-turn metrics catch drift, redundant retrieval, and cross-turn hallucination that per-turn faithfulness misses.
> - **Production and compliance are load-bearing.** Sessions, rate limits, guardrails, isolation - and for India, Indic embeddings plus DPDP/RBI audit trails and consent, designed in from turn one.

> ✅ **Info**
>
> 🗺️ Where this goes next
> - In Lesson 8.10 (**Agentic RAG**) the reformulate-and-remember loop becomes an agent that decides at runtime whether to retrieve, re-query, or answer - step 5’s `create_agent` taken all the way.
> - In Lesson 8.11 (**RAG Evaluation**) step 9’s DeepEval turns into a golden-set + CI harness that gates every change to the system.
> - In Module 14 (**LLMOps**) the session store and per-turn stats become monitored production telemetry - traces, dashboards, and alerts on drift.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or operates one layer of the conversational-RAG stack.

---

## 🎓 Summary

You've completed the practical part of **Conversational RAG- Memory + Retrieval**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_8.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.8.html` - regenerate this notebook whenever the source HTML is updated.*
